# 第25课 · 梯度（gradient）下降（gradient descent，GD） — 用一条直线拟合数据，从损失函数（loss function）到权重更新（weight update）公式

**目标**：要让函数变小，就朝梯度的**反方向**走一小步，反复迭代。

`新位置 = 旧位置 − 学习率（learning rate，lr） × 梯度`

**为什么对 Aurora 重要**：`aurora.train` 里每次调用 `optimizer.step()`，执行的就是这一行更新规则；学习率设错，收敛曲线就会在极值点两侧震荡或长期停滞。

← **上一课**　[L24 · 链式法则](L24_chain_rule.ipynb)

> 上节课学习了 **链式法则**：函数套函数的求导，反向传播的数学基础。  
> 本课将探讨 **梯度下降**。

## 本课剧情：蒙眼下山

你被蒙上眼睛，站在山坡某处，目标是走到山谷。

你能做的：用脚探测当前位置的坡度（=梯度），然后沿最陡下坡方向（=负梯度）走一步。重复这个过程，你就会越来越接近谷底。

这就是**梯度下降**（gradient descent）：

```
x ← x - lr · ∇f(x)
```

- `x`：当前位置（权重向量）
- `∇f(x)`：当前坡度（梯度）
- `lr`：步长（学习率，learning rate）

三个核心问题：
1. **为什么是"负"梯度？** 梯度指向上坡，走反方向才是下坡。
2. **lr 太大会怎样？** 步子太大，可能跨过谷底反弹到另一侧（震荡/发散）。
3. **何时停下？** 梯度接近零时（∇f(x) ≈ 0）——站在了平地（可能是谷底，也可能是鞍点）。

本课实现一步更新 `gd_step(x, grad_value, lr)` 并验证：  
50步后 f(x)=(x-3)² 从 x=0 收敛到 x≈3。

## 1. 手算一步：f(x) = (x−3)²

**解析梯度**：f'(x) = 2(x−3)

**出发点**：x₀ = 0，学习率 lr = 0.1

**第一步更新**：
```
x₁ = x₀ - lr · f'(x₀)
   = 0 - 0.1 · f'(0)
   = 0 - 0.1 · 2·(0-3)
   = 0 - 0.1 · (-6)
   = 0.6
```

**第二步**：
```
x₂ = x₁ - 0.1 · f'(x₁)
   = 0.6 - 0.1 · 2·(0.6-3)
   = 0.6 - 0.1 · (-4.8)
   = 1.08
```

每步后 x 更接近极小值点 x*=3，步长按公比 `(1 - 2·lr)` = 0.8 几何收缩。

**lr 的边界条件**：
- 0 < lr < 1：收敛（|1 - 2·lr| < 1）
- lr = 1：两步内到达（x₁=6, x₂=3）
- lr > 1：发散（|1 - 2·lr| > 1，误差不断放大）

## 实验入口：用数值变化观察函数

三个关键变量：`x`（当前位置）、`grad`（该点梯度值）、`lr`（学习率）。观察每轮迭代后 `x` 怎样靠近 3，以及 `f(x)` 怎样收敛到 0。

In [ ]:
import numpy as np
grad = lambda x: 2*(x-3)      # f'(x)
x = 0.0; lr = 0.1
for step in range(30):
    x = x - lr * grad(x)
print('收敛到 x =', round(x, 4), ' (最小值在 x=3)')

## 动手观察：变化率不是一句口号

先在单个点上算梯度和一步更新，确认数值正确，再进入多步迭代。

In [ ]:
import numpy as np

def f(x):
    return x**2

xs = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
h = 1e-3
slopes = (f(xs + h) - f(xs - h)) / (2 * h)

print('x =', xs)
print('f(x) =', f(xs))
print('近似斜率 =', np.round(slopes, 3))
print('理论斜率 2x =', 2 * xs)


## 代码实验：遍历不同位置，看斜率如何变化

把 `x` 从 −3 遍历到 3，直接打印每个点的函数值和斜率，看清楚梯度在极值点两侧的符号变化。

In [ ]:
import numpy as np

def f(x):
    return x**2 + 2*x

h = 1e-4
for x in np.linspace(-3, 3, 7):
    slope = (f(x + h) - f(x - h)) / (2*h)
    print(f'x={x:5.2f} | f(x)={f(x):6.2f} | slope≈{slope:6.2f}')


## 2. ✏️ 实现一步更新 `gd_step(x, grad_value, lr)`

**推理路线**：
1. `grad_value` 指向函数上升最快的方向；要让函数值下降，就从当前位置减去它：`x - something`。
2. 步长由 `lr` 控制：实际移动量是 `lr * grad_value`，而不是直接减去整个梯度（梯度数值可能很大）。
3. 合并得到：`x_new = x - lr * grad_value`，整个函数只需一行，不需要循环。

**参考输入输出**：`f(x)=(x−3)²`，当前 `x=0`，梯度 `2(0−3)=−6`，`lr=0.1` → 新 `x = 0 − 0.1×(−6) = 0.6`（向 3 靠近）。

> 💡 需要提示？完成练习后可参考 `solutions/` 目录中的参考实现。


### 写代码前，先把变量表补完整

写 `gd_step` 前明确三件事：
- 输入：`x`（当前位置，float）、`grad_value`（该点梯度值）、`lr`（学习率，控制步长大小）
- 关键步骤：将移动量（`lr` 乘以梯度方向）从当前位置**减去**，得到新位置
- 返回：执行一步梯度下降后的新位置


In [ ]:
def gd_step(x, grad_value, lr):
    # ✏️ TODO: 返回更新后的 x
    raise NotImplementedError("请实现 gd_step：返回 x 减去 lr 乘以梯度的新位置")


In [ ]:
# 单步验证 — 先实现 gd_step 再运行此格
# 预期：gd_step(0.0, -6.0, 0.1) = 0.0 - 0.1*(-6.0) = 0.6
try:
    result = gd_step(0.0, -6.0, 0.1)
    assert abs(result - 0.6) < 1e-9, f"单步验证失败：期望 0.6，得到 {result}"
    print('✅ 单步验证通过：gd_step(0.0, -6.0, 0.1) =', result)
except NotImplementedError as e:
    print('⚠️', e)


In [ ]:
try:
    x = 0.0; lr = 0.1
    for _ in range(50):
        x = gd_step(x, 2*(x-3), lr)
    assert abs(x - 3.0) < 1e-3
    print('✅ 通过：你实现了梯度下降的一步，迭代后 x =', round(x, 4))
except NotImplementedError as e:
    print('⚠️', e)


## 3. 实战预告：用梯度下降拟合一条直线 y = 2x + 1（带噪声）

（这一格直接运行看效果，是 Month 2 线性回归（linear regression）的雏形）

In [ ]:
import matplotlib.pyplot as plt
rng = np.random.default_rng(0)
X = np.linspace(0, 5, 40)
y = 2*X + 1 + rng.normal(0, 0.5, size=X.shape)   # 真实关系 + 噪声
w, b, lr = 0.0, 0.0, 0.01
for _ in range(2000):
    pred = w*X + b
    err = pred - y
    w -= lr * np.mean(err * X) * 2     # 对 MSE 的梯度
    b -= lr * np.mean(err) * 2
print(f'学到 w={w:.2f} (真值2), b={b:.2f} (真值1)')
plt.scatter(X, y, s=12, label='data')
plt.plot(X, w*X+b, 'r', label='fit')
plt.legend(); plt.title('linear regression via gradient descent'); plt.show()

**🔗 Aurora 连接**：本节实现的 `gd_step(x, grad_value, lr)` 封装了梯度下降的核心更新逻辑。当 Month 2 引入训练循环时，此更新规则将以 `optimizer.step()` 的形式出现，把「预测 → 算误差 → 沿梯度更新权重」这套逻辑推广到神经网络参数张量，支撑 ASR、音乐生成和 LLM 的全程训练。


## 参数实验：lr 大小决定收敛还是震荡

从 `x=-2.0` 出发，对 `f(x)=(x−3)²` 用 `lr=0.2` 跑 8 步，打印每步的 `x` 值：

- `lr=0.2`：步子适中，`x` 单调逼近极值点 3，8 步内可见明显收敛趋势。

改变 `lr` 观察不同行为：`lr=0.9` 步子较大，`x` 每步越过极值点 3 但振幅递减（序列仍收敛，lr<1 时）；`lr=0.01` 步子很小，收敛稳定但需要更多步。亲眼看清学习率调参对收敛行为的决定性影响。

In [ ]:
def f(x):
    return (x - 3)**2

# 对比四种学习率的收敛行为
# lr<1 且 |1-2*lr|<1（即 0<lr<1）时收敛；lr=1.1 时 |1-2*1.1|=1.2>1，序列发散
for lr in [0.01, 0.2, 0.9, 1.1]:
    x = -2.0
    for step in range(8):
        grad = 2 * (x - 3)
        x = x - lr * grad
    fx = f(x)
    status = '✅ 收敛' if abs(x - 3) < 5 else '❌ 发散'
    print(f'lr={lr:.2f} | 8步后 x={x:9.3f} | f(x)={fx:10.2f} | {status}')


## 本课收束

现在能调用 `gd_step(x, grad_value, lr)` 逐步更新 `x`，在 `f(x) = (x−3)²` 上验证收敛过程。这对应 `aurora.train` 训练循环里每一步的参数更新。下一节（L26）将通过 cviz 等高线图展示梯度下降的收敛轨迹，直观对比不同学习率和起点下的收敛速度。

In [ ]:
# ── 参数实验：学习率对收敛的影响 ─────────────────────────────────────────
import matplotlib.pyplot as plt

learning_rates = [0.001, 0.01, 0.1, 0.5, 1.0]
colors = ['#e74c3c', '#e67e22', '#2ecc71', '#3498db', '#9b59b6']

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for lr, color in zip(learning_rates, colors):
    # 最小化 f(w) = (w - 3)^2，梯度 = 2*(w - 3)，最优解 w=3
    w, losses = 0.0, []
    for _ in range(100):
        grad = 2 * (w - 3.0)
        w -= lr * grad
        losses.append((w - 3.0) ** 2)
    axes[0].plot(losses, label=f'lr={lr}', color=color)

axes[0].set_xlabel('迭代次数'); axes[0].set_ylabel('损失 (w-3)²')
axes[0].set_title('不同学习率的收敛曲线'); axes[0].legend(); axes[0].set_yscale('log')
axes[0].set_ylim(bottom=1e-10)

# 100 步后各学习率的最终损失
final_losses = []
for lr in learning_rates:
    w = 0.0
    for _ in range(100):
        w -= lr * 2 * (w - 3.0)
    final_losses.append((w - 3.0) ** 2)

axes[1].bar([str(lr) for lr in learning_rates], final_losses, color=colors)
axes[1].set_xlabel('学习率'); axes[1].set_ylabel('100步后损失')
axes[1].set_title('学习率 vs 最终损失')
plt.tight_layout(); plt.show()

assert final_losses[2] < final_losses[0], "lr=0.1 应比 lr=0.001 收敛更快"
print("✅ 观察：lr 过小→收敛慢；lr 过大→震荡或发散；最优 lr ≈ 0.1–0.5（取决于问题）")

---
⬇️ **通关检验**：收束小结已读；请完成下方白板挑战后再勾选自评。


## ✏️ 白板挑战：梯度下降手算（目标 10 分钟）

盖上屏幕，纸上作答：

**问 1**：f(x) = (x−3)²，f'(x) = ?  
从 x=0 出发，lr=0.1，手算 x₁ 和 x₂。

**问 2**：`gd_step(0.0, -6.0, 0.1)` 应返回多少？

**问 3**：f(x) = (x−3)²，lr=2.0。  
从 x=0 出发，x₁ = ? x₂ = ?  
这会收敛还是发散？（提示：计算比值 |1 - 2·lr|）

**问 4**：f(x, y) = x² + y²，∇f(x,y) = ?  
从 (2, 4) 出发，lr=0.1，一步之后 (x₁, y₁) = ?

推导完成后运行下面格对答案。

In [ ]:
# ✏️ 对答案格
import numpy as np

# 问1：f'(x)=2(x-3), x0=0, lr=0.1
f = lambda x: (x - 3)**2
grad_f = lambda x: 2 * (x - 3)
x0, lr = 0.0, 0.1
x1_expected = x0 - lr * grad_f(x0)   # 0 - 0.1*(-6) = 0.6
x2_expected = x1_expected - lr * grad_f(x1_expected)  # 0.6 - 0.1*(-4.8) = 1.08
assert np.isclose(x1_expected, 0.6, atol=1e-10)
assert np.isclose(x2_expected, 1.08, atol=1e-10)
print(f"Q1 ✅  x₁={x1_expected:.3f}, x₂={x2_expected:.3f}  (向x*=3收敛)")

# 问2：gd_step(0.0, -6.0, 0.1) = 0 - 0.1*(-6) = 0.6
try:
    result = gd_step(0.0, -6.0, 0.1)
    assert np.isclose(result, 0.6, atol=1e-12), f"gd_step 应返回 0.6，得到 {result}"
    print(f"Q2 ✅  gd_step(0.0,-6.0,0.1) = {result}")
except NotImplementedError:
    print("⬜ Q2：请先实现 gd_step()，再运行对答案格")

# 问3：lr=2.0 发散
lr_big = 2.0
x_div = 0.0
xs_div = [x_div]
for _ in range(4):
    x_div = x_div - lr_big * grad_f(x_div)
    xs_div.append(x_div)
ratio = abs(1 - 2 * lr_big)  # = 3 > 1 → 发散
assert ratio > 1, "lr=2时公比应>1"
print(f"Q3 ✅  lr=2→公比|1-2lr|={ratio}，轨迹={[round(v,1) for v in xs_div]}  → 发散")

# 问4：f(x,y)=x²+y², ∇f=(2x,2y), 从(2,4)出发lr=0.1
lr_2d = 0.1
p = np.array([2., 4.])
grad_2d = np.array([2*p[0], 2*p[1]])   # [4, 8]
p1 = p - lr_2d * grad_2d              # [2-0.4, 4-0.8] = [1.6, 3.2]
assert np.allclose(p1, [1.6, 3.2], atol=1e-12)
print(f"Q4 ✅  ∇f(2,4)=[4,8]，一步后 ({p1[0]:.2f},{p1[1]:.2f})，向(0,0)收敛")
print("\n🎉 梯度下降白板挑战通过！神经网络训练的基本更新步骤已内化。")

In [ ]:
# ✏️ 本课自评
l25_review = {
    "gd_update_formula":          None,  # 记住 x←x-lr·∇f(x)？True/False
    "gd_step_implemented":        None,  # gd_step 实现并通过断言？True/False
    "lr_convergence_condition":   None,  # 知道 lr<1 收敛、lr>1 发散的判据？True/False
    "2d_gradient_descent":        None,  # 能对多元函数逐分量做GD更新？True/False
    "whiteboard_passed":          None,  # 白板挑战纸上推导完成？True/False
}

unfilled = [k for k, v in l25_review.items() if v is None]
assert not unfilled, f'还未填写：{unfilled}'
weak = [k for k, v in l25_review.items() if v is False]
if weak:
    print(f'⚠️  需要加强：{weak}')
else:
    print('✅ L25 全部通关！进入 L26：微积分可视化')

---

→ **下一课**　[L26 · 微积分可视化](L26_visual_calculus.ipynb)

> 下节课将学习 **微积分可视化**：切线、等高线与梯度下降轨迹动态演示。